In [1]:
# =====================================================================
# 🔥 QWEN 2.5-3B-INSTRUCT — ZERO-SHOT
# 🔥 FROM SCRATCH
# 🔥 EXACT SAME PROMPT AS GEMMA
# 🔥 TEXT GENERATION + FULL EVALUATION
# 🔥 2359 ASSET SAMPLES + 10 REFERENCES EACH
# =====================================================================


# =====================================================================
# 1️⃣ INSTALL REQUIRED PACKAGES
# =====================================================================

!pip install -q -U \
    transformers \
    datasets \
    accelerate \
    sentencepiece \
    evaluate \
    rouge-score \
    textstat \
    bert-score \
    sacrebleu \
    sacremoses


# =====================================================================
# 2️⃣ IMPORT LIBRARIES
# =====================================================================

import os
import gc
import time
import torch
import pandas as pd
import numpy as np

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

import evaluate
import textstat


# =====================================================================
# 3️⃣ SYSTEM INFORMATION
# =====================================================================

print("=" * 100)
print("🔥 QWEN 2.5-3B-INSTRUCT — ZERO-SHOT")
print("🔥 EXACT SAME PROMPT AS GEMMA")
print("=" * 100)

print("\nPyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    total_vram = (
        torch.cuda.get_device_properties(0)
        .total_memory
        / (1024 ** 3)
    )

    print(
        "Total VRAM:",
        round(total_vram, 2),
        "GB"
    )


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 61.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 11.3 MB/s eta 0:00:00
🔥 QWEN 2.5-3B-INSTRUCT — ZERO-SHOT
🔥 EXACT SAME PROMPT AS GEMMA

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
Total VRAM: 14.56 GB


In [2]:
# =====================================================================
# 4️⃣ LOAD ASSET DATASET
# =====================================================================

print("\n" + "=" * 100)
print("📂 LOADING ASSET DATASET")
print("=" * 100)

asset = load_dataset("facebook/asset")

asset_df = pd.concat(
    [
        asset["validation"].to_pandas(),
        asset["test"].to_pandas()
    ],
    ignore_index=True
)

print("\nUnique source sentences:", len(asset_df))

print(
    "Total human references:",
    asset_df["simplifications"]
    .apply(len)
    .sum()
)


📂 LOADING ASSET DATASET



Unique source sentences: 2359
Total human references: 23590


In [3]:

# =====================================================================
# 5️⃣ VERIFY DATASET
# =====================================================================

assert len(asset_df) == 2359

assert (
    asset_df["simplifications"]
    .apply(len)
    .eq(10)
    .all()
)

print("\n✅ ASSET verification passed.")
print("Sources:", len(asset_df))

print(
    "References:",
    asset_df["simplifications"]
    .apply(len)
    .sum()
)

print("References per sample: 10")



✅ ASSET verification passed.
Sources: 2359
References: 23590
References per sample: 10


In [4]:
# =====================================================================
# 6️⃣ LOAD QWEN MODEL
# =====================================================================

print("\n" + "=" * 100)
print("🤖 LOADING QWEN MODEL")
print("=" * 100)

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

print("\n✅ Qwen model ready")
print("Model:", MODEL_NAME)

print(
    "Tokenizer:",
    tokenizer.__class__.__name__
)

print(
    "Model:",
    model.__class__.__name__
)


# =====================================================================
# 7️⃣ EXACT GEMMA ZERO-SHOT PROMPT
# ⚠️ DO NOT CHANGE THIS PROMPT
# =====================================================================

def zero_shot_prompt(sentence):

    return f"""Simplify the following sentence using clear and easy-to-understand language.

Preserve the original meaning.
Do not add explanations or commentary.
Return only the simplified sentence.

Original sentence:
{sentence}

Simplified sentence:"""


print("\n" + "=" * 100)
print("📝 ZERO-SHOT PROMPT")
print("=" * 100)

print(
    zero_shot_prompt(
        "This is a test sentence."
    )
)

print("\n✅ Exact Gemma zero-shot prompt ready.")


🤖 LOADING QWEN MODEL


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]


✅ Qwen model ready
Model: Qwen/Qwen2.5-3B-Instruct
Tokenizer: Qwen2Tokenizer
Model: Qwen2ForCausalLM

📝 ZERO-SHOT PROMPT
Simplify the following sentence using clear and easy-to-understand language.

Preserve the original meaning.
Do not add explanations or commentary.
Return only the simplified sentence.

Original sentence:
This is a test sentence.

Simplified sentence:

✅ Exact Gemma zero-shot prompt ready.


In [5]:

# =====================================================================
# 8️⃣ PREPARE QWEN CHAT INPUT
# =====================================================================
# The PROMPT TEXT remains exactly identical to Gemma.
# We only use Qwen's native chat formatting for the model input.
# =====================================================================

def prepare_qwen_input(sentence):

    prompt = zero_shot_prompt(sentence)

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    return text


print("\n✅ Qwen input formatting ready.")



✅ Qwen input formatting ready.


In [6]:
# =====================================================================
# 9️⃣ LOAD EVALUATION METRICS
# =====================================================================

print("\n" + "=" * 100)
print("📊 LOADING EVALUATION METRICS")
print("=" * 100)

sari_metric = evaluate.load(
    "sari"
)

bleu_metric = evaluate.load(
    "bleu"
)

rouge_metric = evaluate.load(
    "rouge"
)

bertscore_metric = evaluate.load(
    "bertscore"
)

print("\n✅ All metrics loaded successfully.")




📊 LOADING EVALUATION METRICS

✅ All metrics loaded successfully.


In [7]:
# =====================================================================
# 🔟 EVALUATE ONE SAMPLE
# =====================================================================

def evaluate_one_sample(
    source,
    prediction,
    references
):

    prediction = str(
        prediction
    ).strip()


    # ================================================================
    # EMPTY PREDICTION SAFETY
    # ================================================================

    if not prediction:

        return {
            "sari": 0.0,
            "bleu": 0.0,
            "rougeL": 0.0,
            "bertscore_f1": 0.0,
            "fkgl": 0.0
        }


    # ================================================================
    # SARI
    # Uses all 10 references
    # ================================================================

    try:

        sari_result = sari_metric.compute(
            sources=[source],
            predictions=[prediction],
            references=[references]
        )

        sari = sari_result["sari"]

    except Exception:

        sari = 0.0


    # ================================================================
    # BLEU
    # Uses all 10 references
    # ================================================================

    try:

        bleu_result = bleu_metric.compute(
            predictions=[prediction],
            references=[references],
            smooth=True
        )

        bleu = bleu_result["bleu"]

    except Exception:

        bleu = 0.0


    # ================================================================
    # ROUGE-L
    # Calculate against each of 10 references
    # Then take mean
    # ================================================================

    rouge_scores = []

    for reference in references:

        try:

            rouge_result = rouge_metric.compute(
                predictions=[prediction],
                references=[reference]
            )

            rouge_scores.append(
                rouge_result["rougeL"]
            )

        except Exception:

            rouge_scores.append(0.0)


    rouge_l = float(
        np.mean(rouge_scores)
    )


    # ================================================================
    # BERTSCORE
    # Compare prediction with all 10 references
    # Then take mean F1
    # ================================================================

    try:

        bert_result = bertscore_metric.compute(

            predictions=[
                prediction
            ] * len(references),

            references=references,

            lang="en"
        )

        bertscore_f1 = float(
            np.mean(
                bert_result["f1"]
            )
        )

    except Exception:

        bertscore_f1 = 0.0


    # ================================================================
    # FKGL
    # ================================================================

    try:

        fkgl = textstat.flesch_kincaid_grade(
            prediction
        )

    except Exception:

        fkgl = 0.0


    # ================================================================
    # RETURN RESULTS
    # ================================================================

    return {

        "sari": float(sari),

        "bleu": float(bleu),

        "rougeL": float(rouge_l),

        "bertscore_f1": float(
            bertscore_f1
        ),

        "fkgl": float(fkgl)
    }


print("\n✅ evaluate_one_sample() ready.")



✅ evaluate_one_sample() ready.


In [8]:
# =====================================================================
# 1️⃣1️⃣ TEST GENERATION
# =====================================================================

print("\n" + "=" * 100)
print("🧪 TEST GENERATION")
print("=" * 100)

example = asset_df.iloc[0]

source = example["original"]

formatted_prompt = prepare_qwen_input(
    source
)

inputs = tokenizer(
    formatted_prompt,
    return_tensors="pt"
).to(model.device)


if torch.cuda.is_available():

    torch.cuda.empty_cache()

    torch.cuda.reset_peak_memory_stats()


start_time = time.perf_counter()


with torch.no_grad():

    output_ids = model.generate(

        **inputs,

        max_new_tokens=128,

        do_sample=False,

        pad_token_id=tokenizer.eos_token_id
    )


end_time = time.perf_counter()


inference_time = (
    end_time - start_time
)


# Remove input prompt tokens

generated_ids = output_ids[
    0,
    inputs["input_ids"].shape[1]:
]


prediction = tokenizer.decode(
    generated_ids,
    skip_special_tokens=True
).strip()


print("\nORIGINAL:")
print(source)

print("\nQWEN GENERATED:")
print(prediction)

print(
    "\nInference time:",
    round(inference_time, 3),
    "seconds"
)


# Clean test variables

del inputs
del output_ids
del generated_ids

gc.collect()

if torch.cuda.is_available():

    torch.cuda.empty_cache()


🧪 TEST GENERATION

ORIGINAL:
Adjacent counties are Marin (to the south), Mendocino (to the north), Lake (northeast), Napa (to the east), and Solano and Contra Costa (to the southeast).

QWEN GENERATED:
The counties next to each other are Marin (south), Mendocino (north), Lake (northeast), Napa (east), Solano, and Contra Costa (southeast).

Inference time: 5.196 seconds


In [9]:

# =====================================================================
# 1️⃣2️⃣ DELETE OLD INCORRECT RESULTS
# =====================================================================

OUTPUT_FILE = (
    "QWEN_ZERO_SHOT_ASSET_2359_FINAL.csv"
)

CHECKPOINT_EVERY = 25


if os.path.exists(OUTPUT_FILE):

    os.remove(OUTPUT_FILE)

    print(
        "\n🗑️ Old Qwen zero-shot results deleted."
    )

else:

    print(
        "\n🆕 No old checkpoint found."
    )


results_df = pd.DataFrame()

completed_ids = set()


print(
    "\n🚀 Starting Qwen Zero-Shot from scratch."
)

print(
    "Total samples:",
    len(asset_df)
)


# =====================================================================
# 1️⃣3️⃣ FULL GENERATION + EVALUATION LOOP
# =====================================================================

for i in range(len(asset_df)):


    # ================================================================
    # SAMPLE HEADER
    # ================================================================

    print("\n")
    print("=" * 100)

    print(
        f"🧪 SAMPLE {i + 1}/"
        f"{len(asset_df)}"
    )

    print("=" * 100)


    # ================================================================
    # DATA
    # ================================================================

    example = asset_df.iloc[i]

    source = example["original"]

    references = example[
        "simplifications"
    ]


    # ================================================================
    # DISPLAY ORIGINAL
    # ================================================================

    print("\n🔵 ORIGINAL:")

    print(source)


    # ================================================================
    # EXACT GEMMA PROMPT
    # QWEN CHAT FORMATTING ONLY
    # ================================================================

    formatted_prompt = prepare_qwen_input(
        source
    )


    # ================================================================
    # TOKENIZATION
    # ================================================================

    inputs = tokenizer(

        formatted_prompt,

        return_tensors="pt"

    ).to(model.device)


    # ================================================================
    # RESET VRAM MEASUREMENT
    # ================================================================

    if torch.cuda.is_available():

        torch.cuda.empty_cache()

        torch.cuda.reset_peak_memory_stats()


    # ================================================================
    # GENERATION
    # ================================================================

    start_time = time.perf_counter()


    with torch.no_grad():

        output_ids = model.generate(

            **inputs,

            max_new_tokens=128,

            do_sample=False,

            pad_token_id=tokenizer.eos_token_id
        )


    end_time = time.perf_counter()


    inference_time = (
        end_time - start_time
    )


    # ================================================================
    # REMOVE PROMPT TOKENS
    # ================================================================

    generated_ids = output_ids[

        0,

        inputs["input_ids"].shape[1]:

    ]


    # ================================================================
    # DECODE GENERATED TEXT
    # ================================================================

    prediction = tokenizer.decode(

        generated_ids,

        skip_special_tokens=True

    ).strip()


    # ================================================================
    # PEAK VRAM
    # Measured immediately after generation
    # Before evaluation metrics
    # ================================================================

    if torch.cuda.is_available():

        peak_vram = (

            torch.cuda.max_memory_allocated()

            / (1024 ** 3)

        )

    else:

        peak_vram = None


    # ================================================================
    # DISPLAY GENERATED TEXT
    # ================================================================

    print("\n🟢 QWEN GENERATED TEXT:")

    print(prediction)


    # ================================================================
    # EVALUATION
    # ================================================================

    scores = evaluate_one_sample(

        source,

        prediction,

        references
    )


    # ================================================================
    # DISPLAY METRICS
    # ================================================================

    print("\n📊 METRICS")

    print("-" * 60)

    print(
        f"SARI          : "
        f"{scores['sari']:.4f}"
    )

    print(
        f"BLEU          : "
        f"{scores['bleu']:.4f}"
    )

    print(
        f"ROUGE-L       : "
        f"{scores['rougeL']:.4f}"
    )

    print(
        f"BERTScore-F1  : "
        f"{scores['bertscore_f1']:.4f}"
    )

    print(
        f"FKGL          : "
        f"{scores['fkgl']:.4f}"
    )


    # ================================================================
    # DISPLAY PERFORMANCE
    # ================================================================

    print("\n⚡ PERFORMANCE")

    print("-" * 60)

    print(
        f"Inference Time: "
        f"{inference_time:.3f} sec"
    )

    if peak_vram is not None:

        print(
            f"Peak VRAM     : "
            f"{peak_vram:.3f} GB"
        )


    # ================================================================
    # CREATE RESULT
    # ================================================================

    result = {

        "sample_id": i,

        "model": "Qwen2.5-3B-Instruct",

        "prompting": "zero-shot",

        "original": source,

        "generated": prediction,

        "references": " ||| ".join(
            references
        ),

        "sari": scores["sari"],

        "bleu": scores["bleu"],

        "rougeL": scores["rougeL"],

        "bertscore_f1": scores[
            "bertscore_f1"
        ],

        "fkgl": scores["fkgl"],

        "inference_time_sec": inference_time,

        "peak_vram_gb": peak_vram
    }


    # ================================================================
    # APPEND RESULT
    # ================================================================

    results_df = pd.concat(

        [

            results_df,

            pd.DataFrame(
                [result]
            )

        ],

        ignore_index=True
    )


    completed_ids.add(i)


    # ================================================================
    # CHECKPOINT EVERY 25 SAMPLES
    # ================================================================

    if (

        len(completed_ids)
        % CHECKPOINT_EVERY == 0

        or

        i == len(asset_df) - 1

    ):

        results_df.to_csv(

            OUTPUT_FILE,

            index=False
        )

        print("\n💾 CHECKPOINT SAVED")

        print(

            f"Completed: "
            f"{len(completed_ids)}/"
            f"{len(asset_df)}"

        )

        print(
            "File:",
            OUTPUT_FILE
        )


    # ================================================================
    # PROGRESS
    # ================================================================

    print("\n📈 PROGRESS")

    print(

        f"[{i + 1}/{len(asset_df)}] "

        f"SARI={scores['sari']:.4f} | "

        f"BLEU={scores['bleu']:.4f} | "

        f"ROUGE-L={scores['rougeL']:.4f} | "

        f"BERT={scores['bertscore_f1']:.4f} | "

        f"FKGL={scores['fkgl']:.4f} | "

        f"Time={inference_time:.2f}s | "

        f"VRAM={peak_vram:.3f}GB"

    )


    # ================================================================
    # CLEANUP
    # ================================================================

    del inputs

    del output_ids

    del generated_ids


    if i % 10 == 0:

        gc.collect()

        if torch.cuda.is_available():

            torch.cuda.empty_cache()


# =====================================================================
# 1️⃣4️⃣ FINAL SAVE
# =====================================================================

results_df.to_csv(

    OUTPUT_FILE,

    index=False
)


# =====================================================================
# 1️⃣5️⃣ FINAL SUMMARY
# =====================================================================

print("\n")
print("=" * 100)
print("🎉 QWEN ZERO-SHOT EVALUATION COMPLETE")
print("=" * 100)

print(
    "\nTotal samples:",
    len(results_df)
)

print(
    "\nOUTPUT FILE:"
)

print(
    OUTPUT_FILE
)


print("\n📊 AVERAGE RESULTS")

print("-" * 60)

print(
    "Average SARI:",
    round(
        results_df["sari"].mean(),
        4
    )
)

print(
    "Average BLEU:",
    round(
        results_df["bleu"].mean(),
        4
    )
)

print(
    "Average ROUGE-L:",
    round(
        results_df["rougeL"].mean(),
        4
    )
)

print(
    "Average BERTScore-F1:",
    round(
        results_df[
            "bertscore_f1"
        ].mean(),
        4
    )
)

print(
    "Average FKGL:",
    round(
        results_df["fkgl"].mean(),
        4
    )
)

print(
    "Average Inference Time:",
    round(
        results_df[
            "inference_time_sec"
        ].mean(),
        4
    ),
    "seconds"
)

print(
    "Average Peak VRAM:",
    round(
        results_df[
            "peak_vram_gb"
        ].mean(),
        4
    ),
    "GB"
)

print("\n✅ DONE")


🗑️ Old Qwen zero-shot results deleted.

🚀 Starting Qwen Zero-Shot from scratch.
Total samples: 2359


🧪 SAMPLE 1/2359

🔵 ORIGINAL:
Adjacent counties are Marin (to the south), Mendocino (to the north), Lake (northeast), Napa (to the east), and Solano and Contra Costa (to the southeast).

🟢 QWEN GENERATED TEXT:
The counties next to each other are Marin (south), Mendocino (north), Lake (northeast), Napa (east), Solano, and Contra Costa (southeast).


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Streaming output truncated to the last 5000 lines.
Inference Time: 1.218 sec
Peak VRAM     : 6.763 GB

📈 PROGRESS
[2176/2359] SARI=36.3225 | BLEU=0.4776 | ROUGE-L=0.4521 | BERT=0.9366 | FKGL=13.9459 | Time=1.22s | VRAM=6.763GB


🧪 SAMPLE 2177/2359

🔵 ORIGINAL:
While at Kahn he was chief architect for the Fisher Building in 1928.

🟢 QWEN GENERATED TEXT:
Kahn was the chief architect of the Fisher Building in 1928.

📊 METRICS
------------------------------------------------------------
SARI          : 38.7289
BLEU          : 0.6851
ROUGE-L       : 0.6422
BERTScore-F1  : 0.9584
FKGL          : 4.7909

⚡ PERFORMANCE
------------------------------------------------------------
Inference Time: 0.913 sec
Peak VRAM     : 6.762 GB

📈 PROGRESS
[2177/2359] SARI=38.7289 | BLEU=0.6851 | ROUGE-L=0.6422 | BERT=0.9584 | FKGL=4.7909 | Time=0.91s | VRAM=6.762GB


🧪 SAMPLE 2178/2359

🔵 ORIGINAL:
He excuses himself because he has to leave for rehearsal, and he and Dr. Schön leave.

🟢 QWEN GENERATED TEXT:
H

In [10]:
# ============================================================
# QWEN ZERO-SHOT — DESCRIPTIVE STATISTICS
# Mean | Median | Min | Max
# ============================================================

import pandas as pd

OUTPUT_FILE = "QWEN_ZERO_SHOT_ASSET_2359_FINAL.csv"

# Load results
results_df = pd.read_csv(OUTPUT_FILE)

print("=" * 80)
print("📊 QWEN ZERO-SHOT — DESCRIPTIVE STATISTICS")
print("=" * 80)

print(f"\nTotal samples: {len(results_df)}\n")

# Metrics to analyze
columns = [
    "sari",
    "bleu",
    "rougeL",
    "bertscore_f1",
    "fkgl",
    "inference_time_sec",
    "peak_vram_gb"
]

# Calculate statistics
stats_df = pd.DataFrame({
    "Mean": results_df[columns].mean(),
    "Median": results_df[columns].median(),
    "Min": results_df[columns].min(),
    "Max": results_df[columns].max()
})

# Round values
stats_df = stats_df.round(4)

# Rename rows for cleaner output
stats_df.index = [
    "SARI",
    "BLEU",
    "ROUGE-L",
    "BERTScore-F1",
    "FKGL",
    "Inference Time (sec)",
    "Peak VRAM (GB)"
]

print(stats_df.to_string())

print("\n" + "=" * 80)
print("📈 INDIVIDUAL RESULTS")
print("=" * 80)

for metric in stats_df.index:

    row = stats_df.loc[metric]

    print(f"\n{metric}")
    print("-" * 50)
    print(f"Mean   : {row['Mean']}")
    print(f"Median : {row['Median']}")
    print(f"Min    : {row['Min']}")
    print(f"Max    : {row['Max']}")

print("\n" + "=" * 80)
print("✅ DESCRIPTIVE STATISTICS COMPLETE")
print("=" * 80)

📊 QWEN ZERO-SHOT — DESCRIPTIVE STATISTICS

Total samples: 2359

                         Mean   Median      Min      Max
SARI                  42.0307  41.6334  14.7228  80.2756
BLEU                   0.6277   0.6326   0.0877   1.0000
ROUGE-L                0.5679   0.5702   0.1282   0.9560
BERTScore-F1           0.9488   0.9501   0.8683   0.9939
FKGL                   8.8710   8.7567  -1.8400  23.5258
Inference Time (sec)   1.2308   1.1329   0.3438   3.5200
Peak VRAM (GB)         6.7630   6.7631   5.7692   6.7697

📈 INDIVIDUAL RESULTS

SARI
--------------------------------------------------
Mean   : 42.0307
Median : 41.6334
Min    : 14.7228
Max    : 80.2756

BLEU
--------------------------------------------------
Mean   : 0.6277
Median : 0.6326
Min    : 0.0877
Max    : 1.0

ROUGE-L
--------------------------------------------------
Mean   : 0.5679
Median : 0.5702
Min    : 0.1282
Max    : 0.956

BERTScore-F1
--------------------------------------------------
Mean   : 0.9488
Median : 0.

In [11]:
# ============================================================
# QWEN ZERO-SHOT — OVERALL DESCRIPTIVE STATISTICS
# Mean | Median | Min | Max
# ============================================================

import pandas as pd

# ============================================================
# FILE
# ============================================================

OUTPUT_FILE = "QWEN_ZERO_SHOT_ASSET_2359_FINAL.csv"

# ============================================================
# LOAD RESULTS
# ============================================================

results_df = pd.read_csv(OUTPUT_FILE)

print("=" * 80)
print("📊 QWEN ZERO-SHOT — OVERALL DESCRIPTIVE STATISTICS")
print("=" * 80)

print(f"\nTotal samples: {len(results_df)}")

# ============================================================
# METRICS
# ============================================================

columns = [
    "sari",
    "bleu",
    "rougeL",
    "bertscore_f1",
    "fkgl",
    "inference_time_sec",
    "peak_vram_gb"
]

# Check that all required columns exist
missing_columns = [
    col for col in columns
    if col not in results_df.columns
]

if missing_columns:
    print("\n❌ Missing columns:")
    for col in missing_columns:
        print(f"   - {col}")
    raise ValueError("Required metric columns are missing from the CSV.")

# ============================================================
# OVERALL STATISTICS
# ============================================================

stats_df = pd.DataFrame({
    "Mean": results_df[columns].mean(),
    "Median": results_df[columns].median(),
    "Min": results_df[columns].min(),
    "Max": results_df[columns].max()
})

# ============================================================
# CLEAN NAMES
# ============================================================

stats_df.index = [
    "SARI",
    "BLEU",
    "ROUGE-L",
    "BERTScore-F1",
    "FKGL",
    "Inference Time (sec)",
    "Peak VRAM (GB)"
]

# Round
stats_df = stats_df.round(4)

# ============================================================
# DISPLAY OVERALL TABLE
# ============================================================

print("\n" + "=" * 80)
print("📈 OVERALL RESULTS — ALL SAMPLES")
print("=" * 80)

print(stats_df.to_string())

# ============================================================
# INDIVIDUAL METRICS
# ============================================================

print("\n" + "=" * 80)
print("📊 INDIVIDUAL OVERALL METRICS")
print("=" * 80)

for metric in stats_df.index:

    row = stats_df.loc[metric]

    print(f"\n{metric}")
    print("-" * 50)
    print(f"Mean   : {row['Mean']:.4f}")
    print(f"Median : {row['Median']:.4f}")
    print(f"Min    : {row['Min']:.4f}")
    print(f"Max    : {row['Max']:.4f}")

# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("✅ OVERALL DESCRIPTIVE STATISTICS COMPLETE")
print("=" * 80)

📊 QWEN ZERO-SHOT — OVERALL DESCRIPTIVE STATISTICS

Total samples: 2359

📈 OVERALL RESULTS — ALL SAMPLES
                         Mean   Median      Min      Max
SARI                  42.0307  41.6334  14.7228  80.2756
BLEU                   0.6277   0.6326   0.0877   1.0000
ROUGE-L                0.5679   0.5702   0.1282   0.9560
BERTScore-F1           0.9488   0.9501   0.8683   0.9939
FKGL                   8.8710   8.7567  -1.8400  23.5258
Inference Time (sec)   1.2308   1.1329   0.3438   3.5200
Peak VRAM (GB)         6.7630   6.7631   5.7692   6.7697

📊 INDIVIDUAL OVERALL METRICS

SARI
--------------------------------------------------
Mean   : 42.0307
Median : 41.6334
Min    : 14.7228
Max    : 80.2756

BLEU
--------------------------------------------------
Mean   : 0.6277
Median : 0.6326
Min    : 0.0877
Max    : 1.0000

ROUGE-L
--------------------------------------------------
Mean   : 0.5679
Median : 0.5702
Min    : 0.1282
Max    : 0.9560

BERTScore-F1
--------------------------

In [12]:
# ============================================================
# QWEN ZERO-SHOT — OVERALL DESCRIPTIVE STATISTICS
# Mean | Median | Min | Max | Std
# ============================================================

import pandas as pd

OUTPUT_FILE = "QWEN_ZERO_SHOT_ASSET_2359_FINAL.csv"

# ============================================================
# LOAD RESULTS
# ============================================================

df = pd.read_csv(OUTPUT_FILE)

print("=" * 80)
print("📊 QWEN ZERO-SHOT — OVERALL DESCRIPTIVE STATISTICS")
print("=" * 80)

print(f"\nTotal Samples: {len(df)}")

# ============================================================
# METRICS
# ============================================================

metrics = [
    "sari",
    "bleu",
    "rougeL",
    "bertscore_f1",
    "fkgl"
]

# ============================================================
# CALCULATE STATISTICS
# ============================================================

stats = pd.DataFrame({
    "Metric": metrics,
    "Mean": [df[m].mean() for m in metrics],
    "Median": [df[m].median() for m in metrics],
    "Min": [df[m].min() for m in metrics],
    "Max": [df[m].max() for m in metrics],
    "Std": [df[m].std() for m in metrics]
})

# ============================================================
# ROUND VALUES
# ============================================================

stats = stats.round(6)

# ============================================================
# DISPLAY
# ============================================================

print("\n" + "=" * 80)
print("📈 DESCRIPTIVE STATISTICS")
print("=" * 80)

print(stats.to_string(index=False))

print("\n" + "=" * 80)
print("✅ DESCRIPTIVE STATISTICS COMPLETE")
print("=" * 80)

📊 QWEN ZERO-SHOT — OVERALL DESCRIPTIVE STATISTICS

Total Samples: 2359

📈 DESCRIPTIVE STATISTICS
      Metric      Mean    Median       Min       Max      Std
        sari 42.030653 41.633407 14.722844 80.275632 8.895504
        bleu  0.627688  0.632588  0.087749  1.000000 0.212812
      rougeL  0.567896  0.570237  0.128231  0.955971 0.132717
bertscore_f1  0.948786  0.950094  0.868290  0.993929 0.017292
        fkgl  8.871049  8.756667 -1.840000 23.525806 3.506354

✅ DESCRIPTIVE STATISTICS COMPLETE
